In [1]:
import sys

project_root = "/Users/macbookair/Documents/wb_logistics_support"
if project_root not in sys.path:
    sys.path.append(project_root)

In [2]:
# Run schema.sql to create tables (only needed once)
import subprocess
result = subprocess.run(
    ["psql", "wb_logistics", "-f", "../db/schema.sql"],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)

CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE

psql:../db/schema.sql:9: NOTICE:  relation "warehouses" already exists, skipping
psql:../db/schema.sql:17: NOTICE:  relation "warehouse_remains" already exists, skipping
psql:../db/schema.sql:45: NOTICE:  relation "paid_storage" already exists, skipping
psql:../db/schema.sql:58: NOTICE:  relation "region_sale" already exists, skipping
psql:../db/schema.sql:71: NOTICE:  relation "goods_return" already exists, skipping



In [3]:
from src.ingestion.wb_reports import (
    get_warehouses,
    get_warehouse_remains_report,
    get_paid_storage_report,
    get_region_sale,
    get_goods_return,
)
from src.preprocessing.normalize import (
    load_warehouses,
    load_warehouse_remains,
    load_paid_storage,
    load_region_sale,
    load_goods_return,
)

In [5]:
# Warehouses
data = get_warehouses()
rows = load_warehouses(data)
print(f"Loaded {rows} rows into warehouses")

HTTPError: 429 Client Error: Too Many Requests for url: https://supplies-api.wildberries.ru/api/v1/warehouses

In [4]:
# Warehouse Remains
data = get_warehouse_remains_report(locale="ru", groupByNm=True)
rows = load_warehouse_remains(data)
print(f"Loaded {rows} rows into warehouse_remains")

Создан отчёт, taskId = a5c86bd3-30e5-4548-9226-53aec37007aa
Loaded 88 rows into warehouse_remains


In [5]:
# Paid Storage (max 7 days per request)
data = get_paid_storage_report(date_from="2026-03-01", date_to="2026-03-07")
rows = load_paid_storage(data)
print(f"Loaded {rows} rows into paid_storage")

Создан отчёт, taskId = 334c8c1d-3df9-4df7-9fb8-cf9040116d00
Loaded 704 rows into paid_storage


In [6]:
# Region Sale
data = get_region_sale(date_from="2026-03-01", date_to="2026-03-31")
rows = load_region_sale(data)
print(f"Loaded {rows} rows into region_sale")

Loaded 25 rows into region_sale


In [7]:
# Goods Return
data = get_goods_return(date_from="2026-03-01", date_to="2026-03-31")
rows = load_goods_return(data)
print(f"Loaded {rows} rows into goods_return")

Loaded 0 rows into goods_return


In [8]:
# Verify data in DB
import pandas as pd
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

load_dotenv()
engine = create_engine(os.getenv("DATABASE_URL"))

for table in ["warehouses", "warehouse_remains", "paid_storage", "region_sale", "goods_return"]:
    df = pd.read_sql(f"SELECT COUNT(*) FROM {table}", engine)
    print(f"{table}: {df.iloc[0,0]} rows")

warehouses: 127 rows
warehouse_remains: 88 rows
paid_storage: 1998 rows
region_sale: 100 rows
goods_return: 0 rows


In [4]:
import time
#  Load full March for paid_storage (7-day chunks)
periods = [
    ("2026-03-01", "2026-03-07"),
    ("2026-03-08", "2026-03-14"),
    ("2026-03-15", "2026-03-21"),
    ("2026-03-22", "2026-03-31"),
]

for date_from, date_to in periods:
    data = get_paid_storage_report(date_from=date_from, date_to=date_to)
    rows = load_paid_storage(data)
    print(f"{date_from} -> {date_to}: {rows} rows")
    time.sleep(60) 


Создан отчёт, taskId = eb923f84-566f-4ff5-8aa9-454e0a403fb8
2026-03-01 -> 2026-03-07: 704 rows
Создан отчёт, taskId = 0e917c2a-edda-4956-ba6a-14871ac1084f
2026-03-08 -> 2026-03-14: 666 rows
Создан отчёт, taskId = 4a4e52d7-bd18-47d4-abf4-f07db327ff0d
2026-03-15 -> 2026-03-21: 649 rows


HTTPError: 400 Client Error: Bad Request for url: https://seller-analytics-api.wildberries.ru/api/v1/paid_storage?dateFrom=2026-03-22&dateTo=2026-03-31